In [1]:
import pandas as pd

df = pd.read_csv('events.csv')
df.head(5)

,id,user_id,sequence_number,session_id,created_at,ip_address,city,state,postal_code,browser,traffic_source,uri,event_type
0,2198523,NaN,3,83889ed2-2adc-4b9a-af5d-154f6998e778,2021-06-17 17:30:00+00:00,138.143.9.202,São Paulo,São Paulo,02675-031,Chrome,Adwords,/cancel,cancel
1,1773216,NaN,3,7a3fc3f2-e84f-44fe-8876-eff76741f7a3,2020-08-07 08:41:00+00:00,85.114.141.79,Santa Isabel,São Paulo,07500-000,Safari,Adwords,/cancel,cancel
2,2380515,NaN,3,13d9b2fb-eee1-43fd-965c-267b38dd7125,2021-02-15 18:48:00+00:00,169.250.255.132,Mairiporã,São Paulo,07600-000,IE,Adwords,/cancel,cancel
3,2250597,NaN,3,96f1d44e-9621-463c-954c-d8deb7fffe7f,2022-03-30 10:56:00+00:00,137.25.222.160,Cajamar,São Paulo,07750-000,Chrome,Adwords,/cancel,cancel
4,1834446,NaN,3,d09dce10-a7cb-47d3-a9af-44975566fa03,2019-09-05 01:18:00+00:00,161.114.4.174,São Paulo,São Paulo,09581-680,Chrome,Email,/cancel,cancel


In [2]:
df.shape

(2431963, 13)

In [3]:
df_copy = df.copy()

In [4]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
import random
import torch
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from collections import Counter
from torch.utils.data import Dataset, DataLoader

In [5]:
df_copy = df_copy.dropna(subset=['user_id'])

#filter product
df_copy = df_copy[df_copy['uri'].str.startswith('/product', na=False)].copy()

#extract product_id
df_copy['product_id'] = df_copy['uri'].str.extract(r'/product/(\d+)')
df_copy = df_copy.dropna(subset=['product_id'])

#sort
df_copy = df_copy.sort_values(['user_id', 'created_at'])

#filter user (chọn user từng xem trên 5 product khác nhaup)
valid_users = df_copy.groupby('user_id').filter(lambda x: x['product_id'].nunique() >= 5)['user_id'].unique()

df_copy = df_copy[df_copy['user_id'].isin(valid_users)]

In [6]:
#gộp các product_id giống nhau trong khoảng thời gian ngắn liền kề (p1 -> p1 -> p2 -> p2 thành p1 -> p2)
df_copy = df_copy[~(df_copy['product_id'] == df_copy.groupby('user_id')['product_id'].shift())]

In [7]:
df_copy.head(20)

,id,user_id,sequence_number,session_id,created_at,ip_address,city,state,postal_code,browser,traffic_source,uri,event_type,product_id
439244,48,3.0,3,a07ed72e-7d80-48a6-a2a4-1a9b5478e1c9,2023-03-10 07:10:33+00:00,7.14.75.96,Hallandale Beach,Florida,33009,IE,Email,/product/18177,product,18177
663026,59,3.0,2,fd1c1a5c-4dce-485c-a77d-d11d3228a139,2023-04-02 07:39:01+00:00,43.76.98.111,Hallandale Beach,Florida,33009,Firefox,Adwords,/product/28050,product,28050
1209063,52,3.0,2,2afa8d52-63e2-46a1-9ad4-d8fb8381c9a6,2023-04-02 07:42:58+00:00,144.40.156.52,Hallandale Beach,Florida,33009,Safari,Email,/product/21364,product,21364
737588,38,3.0,3,7eebfda1-57ee-45eb-ae36-cf9ade1886b5,2023-04-23 08:59:59+00:00,116.84.193.114,Hallandale Beach,Florida,33009,Firefox,Email,/product/22308,product,22308
2052884,43,3.0,3,3e10d6bb-6d03-48e6-b5c6-40eac8325b88,2023-08-08 06:08:57+00:00,71.100.132.66,Hallandale Beach,Florida,33009,Chrome,Email,/product/26696,product,26696
418168,174,9.0,2,69744cff-3ba2-48a8-a020-4c85f9dda17d,2020-05-23 13:47:24+00:00,25.127.144.138,Xiamen,Gansu province,730031,Chrome,Adwords,/product/20051,product,20051
641928,187,9.0,2,2f6b5144-b61d-4f41-9546-b3403e2cbeb0,2020-05-23 13:47:43+00:00,187.232.61.11,Xiamen,Gansu province,730031,Chrome,Email,/product/22961,product,22961
21136,177,9.0,5,69744cff-3ba2-48a8-a020-4c85f9dda17d,2020-05-23 13:50:02+00:00,25.127.144.138,Xiamen,Gansu province,730031,Chrome,Adwords,/product/20051,product,20051
1560363,190,9.0,5,2f6b5144-b61d-4f41-9546-b3403e2cbeb0,2020-05-23 13:52:02+00:00,187.232.61.11,Xiamen,Gansu province,730031,Chrome,Email,/product/22961,product,22961
145073,180,9.0,8,69744cff-3ba2-48a8-a020-4c85f9dda17d,2020-05-23 13:53:18+00:00,25.127.144.138,Xiamen,Gansu province,730031,Chrome,Adwords,/product/20051,product,20051


In [8]:
df_copy.shape

(57712, 14)

In [9]:
#split train_df và test_df
train_df = df_copy.groupby('user_id').apply(lambda x: x.iloc[:-1]).reset_index(drop=True)
test_df = df_copy.groupby('user_id').tail(1)

test_df = test_df[test_df['product_id'].isin(train_df['product_id'])]

C:\Users\Admin\AppData\Local\Temp\ipykernel_11176\3473076652.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_df = df_copy.groupby('user_id').apply(lambda x: x.iloc[:-1]).reset_index(drop=True)


In [10]:
test_df.head(50)

,id,user_id,sequence_number,session_id,created_at,ip_address,city,state,postal_code,browser,traffic_source,uri,event_type,product_id
2052884,43,3.0,3,3e10d6bb-6d03-48e6-b5c6-40eac8325b88,2023-08-08 06:08:57+00:00,71.100.132.66,Hallandale Beach,Florida,33009,Chrome,Email,/product/26696,product,26696
344947,308,17.0,2,22234583-852e-43bf-90a6-80ab1715fa8c,2020-10-29 12:06:22+00:00,29.60.204.183,High Wycombe,England,HP13,Safari,Email,/product/29058,product,29058
2274341,515,33.0,2,c9369421-e21c-45fa-8e63-9512efda862d,2021-10-15 01:52:43+00:00,28.22.147.26,Yokohama City,Kanagawa,225-0003,Other,Email,/product/25305,product,25305
1163041,581,39.0,8,3c8ade07-36f1-46bf-bd65-146af54f152e,2022-10-10 21:46:21+00:00,172.226.101.172,São Luís,Maranhão,65063,Chrome,Email,/product/25249,product,25249
17411,714,43.0,3,1b6c1ea6-0ea2-4c49-9b40-12d3ef8525bb,2023-07-03 01:28:06+00:00,88.160.137.136,Shanghai,Jiangxi,334600,Other,Adwords,/product/11205,product,11205
489533,939,60.0,2,3d3ef010-93b5-4833-bade-fc48594324c3,2023-04-22 16:00:51+00:00,92.199.54.75,Putian,Chongqing,400032,Chrome,Adwords,/product/8865,product,8865
964278,1943,139.0,2,3a939086-f674-44e5-9fa8-e1f4fba5e04a,2023-11-02 12:12:25+00:00,111.56.58.213,São Luís,Maranhão,65137-000,Chrome,Email,/product/6343,product,6343
2083081,2084,147.0,5,7e3603f6-fc5d-44a0-95db-59941aa70489,2024-01-15 10:24:01.317841+00:00,159.132.116.100,Portsmouth,England,PO2,Chrome,Adwords,/product/14376,product,14376
2425311,2209,158.0,11,ce77cf0a-61b9-4576-8399-3af4d96da935,2023-02-28 10:43:18+00:00,44.181.107.196,Four-stage,Hunan,425099,Chrome,Adwords,/product/13264,product,13264
14594,2377,173.0,2,b04ac763-327f-4d11-a861-691a192e3d10,2023-12-25 14:18:13+00:00,171.125.67.26,Ziyang,Beijing,100120,Firefox,Adwords,/product/10451,product,10451


In [131]:
user_interactions = train_df.groupby('user_id').size()

avg_interaction = user_interactions.mean()

print("Trung bình tương tác của mỗi user trong tập train:", avg_interaction)

Trung bình tương tác của mỗi user trong tập train: 6.157633635123403


In [137]:
print(user_interactions.describe())

count    8063.000000
mean        6.157634
std         2.654145
min         4.000000
25%         4.000000
50%         5.000000
75%         7.000000
max        24.000000
dtype: float64


In [136]:
product_counts = df_copy.groupby('product_id').size()
products_more_than_2 = (product_counts > 2).sum()
print("Tổng số lượng product:",len(product_counts))
print("Số lượng product được tương tác trên 2 lần:", products_more_than_2)
print("Tỷ lệ:",products_more_than_2/len(product_counts))

Tổng số lượng product: 23389
Số lượng product được tương tác trên 2 lần: 9005
Tỷ lệ: 0.3850100474582068


In [11]:
# encode từ train_df
user2id = {u:i for i,u in enumerate(train_df['user_id'].unique())}
item2id = {i:j for j,i in enumerate(train_df['product_id'].unique())}
id2item = {v:k for k,v in item2id.items()}
id2user = {i: u for u, i in user2id.items()}

train_df['uid'] = train_df['user_id'].map(user2id)
train_df['iid'] = train_df['product_id'].map(item2id)

# test_df map theo id có trong train_df
test_df['uid'] = test_df['user_id'].map(user2id)
test_df['iid'] = test_df['product_id'].map(item2id)

num_users = len(user2id)
num_items = len(item2id)

In [12]:
print(num_users)
print(num_items)

8063
21727


In [13]:
meta_products = pd.read_csv('products.csv')
meta_products.head(5)

,id,cost,category,name,brand,retail_price,department,sku,distribution_center_id
0,13842,2.51875,Accessories,Low Profile Dyed Cotton Twill Cap - Navy W39S55D,MG,6.25,Women,EBD58B8A3F1D72F4206201DA62FB1204,1
1,13928,2.33835,Accessories,Low Profile Dyed Cotton Twill Cap - Putty W39S55D,MG,5.95,Women,2EAC42424D12436BDD6A5B8A88480CC3,1
2,14115,4.87956,Accessories,Enzyme Regular Solid Army Caps-Black W35S45D,MG,10.99,Women,EE364229B2791D1EF9355708EFF0BA34,1
3,14157,4.64877,Accessories,Enzyme Regular Solid Army Caps-Olive W35S45D (...,MG,10.99,Women,00BD13095D06C20B11A2993CA419D16B,1
4,14273,6.50793,Accessories,Washed Canvas Ivy Cap - Black W11S64C,MG,15.99,Women,F531DC20FDE20B7ADF3A73F52B71D0AF,1


In [14]:
meta = meta_products.copy()
meta['id'] = meta['id'].astype(str)
meta['iid'] = meta['id'].map(item2id)
meta = meta.dropna(subset=['iid'])
meta['iid'] = meta['iid'].astype(int)
meta = meta.sort_values(by='iid').reset_index(drop=True)

In [15]:
meta.head(5)

,id,cost,category,name,brand,retail_price,department,sku,distribution_center_id,iid
0,18177,49.825469,Active,Smartwool Men's Midweight Bottom,SmartWool,109.989998,Men,8288EB462E80418E70A393D245A984CE,2,0
1,28050,40.748891,Swim,Volcom Men's Bruce Annihilator Board-Shorts,Volcom,63.970001,Men,A4D35E93D6C0787428F2FDF6A29457E0,4,1
2,21364,28.798000,Jeans,Marc Ecko Cut & Sew Men's Baked Alaska Bootcut...,Marc Ecko Cut & Sew,59.500000,Men,6B387EBBCB8020CE186644D4A4669C6A,1,2
3,22308,30.174971,Pants,Dockers Men's Limited Offer D2 Stretch Khaki Pant,Dockers,59.990002,Men,4B6E16D36F691EEC61154D01871CEC76,7,3
4,20051,110.755736,Suits & Sport Coats,Tommy Hilfiger Men's Slim Stripe Trim Fit Suit,Tommy Hilfiger,259.989990,Men,482BD57EA95BB42CC15C82D63AF42EA9,1,4


In [16]:
user_pos_items = train_df.groupby('uid')['iid'].apply(set).to_dict()

In [ ]:
item_meta = meta.set_index('id', drop=False)[['name', 'category', 'department']]

def show_products(pid_list, title):
    print(f"\n===== {title} =====")

    for pid in pid_list:
        if pid in item_meta.index:
            info = item_meta.loc[pid]

            name_col = 'name' if 'name' in info else list(info.index)[0]

            print(
                f"PID: {pid} | "
                f"Name: {info[name_col]} | "
                f"Category: {info.get('category', 'N/A')} | "
                f"Dept: {info.get('department', 'N/A')}"
            )
        else:
            print(f"PID: {pid} | (no metadata)")

In [ ]:
def show_user_seen_products(uid, df_copy, meta_products, id2user, top_n=10):
    """
    uid: user_id đã encode
    df_copy: event log gốc (user_id thật + product_id thật)
    meta_products: metadata theo product_id thật
    id2user: mapping uid → user_id thật
    top_n: số sản phẩm muốn hiển thị
    """
    user_real = id2user[uid]

    # 2. Lấy toàn bộ event log của user trong df_copy
    user_logs = df_copy[df_copy["user_id"] == user_real]

    # 3. Lấy danh sách product_id thật
    product_real_list = user_logs["product_id"].tolist()

    # 4. Chuẩn bị metadata
    df_copy["product_id"] = df_copy["product_id"].astype(str)
    meta_products["id"] = meta_products["id"].astype(str)
    product_meta = meta_products.set_index("id")

    print(f"\n===== USER {uid} (real id: {user_real}) — SEEN PRODUCTS =====")

    for p in product_real_list[:top_n]:
        if p in product_meta.index:
            info = product_meta.loc[p]
            print(
                f"PID: {p} | "
                f"Name: {info['name']} | "
                f"Category: {info['category']} | "
                f"Dept: {info['department']}"
            )
        else:
            print(f"PID: {p} | (no metadata)")

**ALS-WMF**

In [17]:
rows = train_df['uid'].values 
cols = train_df['iid'].values #danh sách item mà user u đã tương tác trong tập train
data = np.ones(len(train_df)) 

R = csr_matrix((data, (rows, cols)), shape=(num_users, num_items)) #(user u x item i) values 0/1

In [18]:
test_users = test_df['uid'].values
test_items = test_df['iid'].values
data_test = np.ones(len(test_df)) 

R_test = csr_matrix((data_test, (test_users, test_items)), shape=(num_users, num_items))
test_dict = {}
for u, i in zip(test_users, test_items):
    test_dict[u] = i

In [19]:
def auc_user_fast(model, R, num_items, n_neg=100):
    R_csr = R.tocsr()
    aucs = []

    for u in range(model.num_users):
        start, end = R_csr.indptr[u], R_csr.indptr[u+1]
        pos_items = list(R_csr.indices[start:end])

        if len(pos_items) == 0:
            continue

        scores = model.predict(u)

        neg_items = np.random.choice(
            np.setdiff1d(np.arange(num_items), pos_items),
            size=n_neg,
            replace=False
        )

        # pairwise AUC
        auc = 0
        total = 0

        for p in pos_items:
            auc += np.sum(scores[p] > scores[neg_items])
            total += len(neg_items)

        aucs.append(auc / total)

    return np.mean(aucs)

In [20]:
class ImplicitALS:
    def __init__(self, num_factors, reg, epochs, alpha):
        self.num_factors = num_factors
        self.reg = reg
        self.epochs = epochs
        self.alpha = alpha

    def fit(self, R):
        self.R = R
        self.num_users, self.num_items = R.shape #R ~ XY^T

        self.X = np.random.normal(0, 0.1, (self.num_users, self.num_factors))
        self.Y = np.random.normal(0, 0.1, (self.num_items, self.num_factors))

        R_csr = R.tocsr() # user -> item
        R_csc = R.tocsc() # item -> user

        I = np.eye(self.num_factors)

        for epoch in range(self.epochs):
            # update user factors
            for u in range(self.num_users):
                start, end = R_csr.indptr[u], R_csr.indptr[u+1]
                items = R_csr.indices[start:end] #item đã tương tác

                Cu = np.ones(len(items)) * self.alpha #confidence
                Pu = np.ones(len(items)) #positive

                Y_i = self.Y[items] # Y_i = vector item tương ứng

                A = Y_i.T @ (np.diag(Cu - 1)) @ Y_i + Y_i.T @ Y_i + self.reg * I
                b = Y_i.T @ (Cu * Pu)

                self.X[u] = np.linalg.solve(A, b) #update user embedding x_u

            # update item factors
            for i in range(self.num_items):
                start, end = R_csc.indptr[i], R_csc.indptr[i+1]
                users = R_csc.indices[start:end]

                Ci = np.ones(len(users)) * self.alpha
                Pi = np.ones(len(users))

                X_u = self.X[users]

                A = X_u.T @ (np.diag(Ci - 1)) @ X_u + X_u.T @ X_u + self.reg * I
                b = X_u.T @ (Ci * Pi)

                self.Y[i] = np.linalg.solve(A, b)

            train_auc = auc_user_fast(self, self.R, self.num_items) #mức độ fit dữ liệu train
            test_auc = auc_user_fast(self, R_test, self.num_items) #khả năng dự đoán item chưa interact

            print(f"Epoch {epoch+1} | train AUC: {train_auc:.4f} | test AUC: {test_auc:.4f}")

    def predict(self, u):
        return self.X[u] @ self.Y.T

In [89]:
def hit_rate_at_k_ALS(ranked_list, true_item, k):
    return int(true_item in ranked_list[:k])

def ndcg_at_k_ALS(ranked_list, true_item, k):
    if true_item in ranked_list[:k]:
        rank = ranked_list.index(true_item)
        return 1 / np.log2(rank + 2)
    return 0

In [92]:
def evaluate_ALS(model, train_R, test_dict, K=10):
    hits, ndcgs = [], []

    num_items = train_R.shape[1]

    for u in test_dict:
        true_item = test_dict[u]

        scores = model.predict(u)

        # filter seen items
        train_items = set(train_R[u].indices)
        scores[list(train_items)] = -np.inf

        ranked_items = np.argsort(-scores)

        hits.append(hit_rate_at_k_ALS(ranked_items, true_item, K))
        ndcgs.append(ndcg_at_k_ALS(ranked_items.tolist(), true_item, K))

    return np.mean(hits), np.mean(ndcgs)

In [25]:
model_ALS = ImplicitALS(num_factors=128,reg=0.01,epochs=10,alpha=40)
model_ALS.fit(R)

hit, ndcg = evaluate_ALS(model_ALS, R, test_dict, K=10)

Epoch 1 | train AUC: 1.0000 | test AUC: 0.5420
Epoch 2 | train AUC: 1.0000 | test AUC: 0.5417
Epoch 3 | train AUC: 1.0000 | test AUC: 0.5405
Epoch 4 | train AUC: 1.0000 | test AUC: 0.5412
Epoch 5 | train AUC: 1.0000 | test AUC: 0.5412
Epoch 6 | train AUC: 1.0000 | test AUC: 0.5409
Epoch 7 | train AUC: 1.0000 | test AUC: 0.5416
Epoch 8 | train AUC: 1.0000 | test AUC: 0.5419
Epoch 9 | train AUC: 1.0000 | test AUC: 0.5421
Epoch 10 | train AUC: 1.0000 | test AUC: 0.5425


In [27]:
print(hit)
print(ndcg)

0.0003232062055591467
0.00015060758714212003


In [93]:
hit, ndcg = evaluate_ALS(model_ALS, R, test_dict, K=5)
print(hit)
print(ndcg)

0.00016160310277957336
0.00010196020581309915


In [59]:
def recommend_als(model, uid, train_R, K=10):
    scores = model.predict(uid)

    seen_items = set(train_R[uid].indices)
    scores[list(seen_items)] = -np.inf

    top_items = np.argsort(-scores)[:K]
    return top_items

def recommend_als_with_product(model, uid, train_R, id2item, K=10):
    top_items = recommend_als(model, uid, train_R, K)

    return [id2item[i] for i in top_items]

def show_recommendation(model, uid, train_R, id2item, K=10):
    recs = recommend_als_with_product(model, uid, train_R, id2item, K)

    print(f"User {uid} top-{K} recommendations:")
    for rank, item in enumerate(recs, 1):
        print(f"{rank}. {item}")

In [222]:
uid = 0 #num = 64 reg = 0.001 alpha = 40

print("User seen\n", [id2item[i] for i in user_pos_items[uid]])
recs_als = recommend_als_with_product(model_ALS, uid, R, id2item, K=10)
print("Recommend\n", recs_als)

User seen
 ['24171', '28283', '16381', '18612', '23584', '23645']
Recommend
 ['26291', '13442', '20242', '21151', '28205', '18938', '17857', '27865', '15013', '10870']


In [229]:
uid = 0 #num = 128 reg = 0.001 alpha = 20

print("User seen\n", [id2item[i] for i in user_pos_items[uid]])
recs_als = recommend_als_with_product(model_ALS, uid, R, id2item, K=10)
print("Recommend\n", recs_als)

User seen
 ['24171', '28283', '16381', '18612', '23584', '23645']
Recommend
 ['16262', '22332', '14972', '26672', '27237', '10582', '4667', '19193', '29074', '18235']


In [29]:
uid = 0 # num_factors=128,reg=0.01,epochs=10,alpha=40

print("User seen\n", [id2item[i] for i in user_pos_items[uid]])
recs_als = recommend_als_with_product(model_ALS, uid, R, id2item, K=10)
print("Recommend\n", recs_als)

User seen
 ['18177', '28050', '21364', '22308']
Recommend
 ['27456', '23416', '22574', '23344', '21120', '27792', '23037', '17866', '27214', '28878']


In [61]:
show_products(recs_als, "RECOMMENDED PRODUCTS (ALS-WMF)")


===== RECOMMENDED PRODUCTS (ALS-WMF) =====
PID: 27456 | Name: Speedo Men's Mighty Python Xtra Life Lycra Brief Swimsuit | Category: Swim | Dept: Men
PID: 23416 | Name: Ed Garments Men's Flat Front Utility Cargo Short. 2468 | Category: Shorts | Dept: Men
PID: 22574 | Name: American Apparel Stretch Chambray Stripe Travel Pant | Category: Pants | Dept: Men
PID: 23344 | Name: Union Skyway Cargo Shorts (450H) - Red | Category: Shorts | Dept: Men
PID: 21120 | Name: 7 For All Mankind Men's Classic Bootcut Jean (Long Inseam) in New York Dark | Category: Jeans | Dept: Men
PID: 27792 | Name: Nike Tri Unisuit Male | Category: Swim | Dept: Men
PID: 23037 | Name: Mountain Hardwear Men's Refueler Shorts | Category: Shorts | Dept: Men
PID: 17866 | Name: Caterpillar Mens Tm Banner Hoodie | Category: Fashion Hoodies & Sweatshirts | Dept: Men
PID: 27214 | Name: Hunter Green Hooded Terry Robe W/double stitching 50L | Category: Sleep & Lounge | Dept: Men
PID: 28878 | Name: Levi's Brown Tonal Stitched Mon

**BPR-MF**

In [62]:
def auc_user_sample(u, U, V, user_pos_items, num_items, n_samples=100):
    pos_items = user_pos_items.get(u, None)
    
    if pos_items is None or len(pos_items) == 0:
        return None

    pos_items = list(pos_items)
    correct = 0
    total = 0

    for _ in range(n_samples):
        # sample positive item
        i = np.random.choice(pos_items)

        # sample negative item
        while True:
            j = np.random.randint(0, num_items)
            if j not in pos_items:
                break

        # compute scores
        x_ui = np.dot(U[u], V[i])
        x_uj = np.dot(U[u], V[j])

        # compare
        if x_ui > x_uj:
            correct += 1

        total += 1

    return correct / total if total > 0 else None

def auc_all_sample(U, V, user_pos_items, num_items, n_samples=100):
    aucs = []

    for u in user_pos_items.keys():
        auc = auc_user_sample(u, U, V, user_pos_items, num_items, n_samples)
        if auc is not None:
            aucs.append(auc)

    return np.mean(aucs)

In [67]:
dim_BP = 128 #32 dữ liệu ít
lr_BP = 0.01 #0.01
reg_BP = 0.01 #0.02
epochs_BP = 30 #15

U = np.random.normal(0, 0.1, (num_users, dim_BP))
V = np.random.normal(0, 0.1, (num_items, dim_BP))

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sample():
    u = random.choice(list(user_pos_items.keys()))
    i = random.choice(list(user_pos_items[u]))

    # negative sampling
    while True:
        j = random.randint(0, num_items - 1)
        if j not in user_pos_items[u]:
            break

    return u, i, j

In [68]:
for epoch in range(epochs_BP):
    loss = 0

    for _ in range(len(train_df)):
        u, i, j = sample()

        x_ui = U[u] @ V[i]
        x_uj = U[u] @ V[j]

        x_uij = x_ui - x_uj

        s = sigmoid(-x_uij)

        # gradients
        grad_u = s * (V[j] - V[i]) + reg_BP * U[u]
        grad_i = s * (-U[u]) + reg_BP * V[i]
        grad_j = s * (U[u]) + reg_BP * V[j]

        # update
        U[u] -= lr_BP * grad_u
        V[i] -= lr_BP * grad_i
        V[j] -= lr_BP * grad_j

        loss += -np.log(sigmoid(x_uij))

    auc = auc_all_sample(U, V, user_pos_items, num_items, n_samples=50)    
    print(f"--Epoch {epoch+1}, loss: {loss:.4f}--")
    print(f"AUC (test): {auc:.4f}")
    print(f"x_ui is {x_ui:.4f}, x_uj is {x_uj:.4f}\n")

--Epoch 1, loss: 34348.9103--
AUC (test): 0.5408
x_ui is 0.0407, x_uj is -0.0426

--Epoch 2, loss: 33936.3613--
AUC (test): 0.5825
x_ui is -0.1025, x_uj is 0.0696

--Epoch 3, loss: 33586.1332--
AUC (test): 0.6206
x_ui is -0.1021, x_uj is 0.1062

--Epoch 4, loss: 33160.1210--
AUC (test): 0.6588
x_ui is 0.0578, x_uj is 0.0924

--Epoch 5, loss: 32772.5045--
AUC (test): 0.6947
x_ui is 0.1501, x_uj is 0.0901

--Epoch 6, loss: 32425.7314--
AUC (test): 0.7271
x_ui is 0.0076, x_uj is 0.1538

--Epoch 7, loss: 32036.8508--
AUC (test): 0.7555
x_ui is 0.1725, x_uj is -0.1032

--Epoch 8, loss: 31656.3315--
AUC (test): 0.7853
x_ui is 0.0699, x_uj is 0.0166

--Epoch 9, loss: 31319.7108--
AUC (test): 0.8115
x_ui is 0.0805, x_uj is -0.1586

--Epoch 10, loss: 30942.0914--
AUC (test): 0.8346
x_ui is -0.0645, x_uj is -0.0675

--Epoch 11, loss: 30603.7220--
AUC (test): 0.8551
x_ui is 0.0321, x_uj is -0.0636

--Epoch 12, loss: 30231.6218--
AUC (test): 0.8728
x_ui is -0.0120, x_uj is -0.2217

--Epoch 13, los

In [96]:
def predict_user_BPR(u):
    return U[u] @ V.T

def recommend_BPR(u, K=10):
    scores = predict_user_BPR(u)

    # loại item đã seen
    seen = user_pos_items[u]
    scores[list(seen)] = -np.inf

    top_k = np.argsort(scores)[-K:][::-1]
    return top_k

def recommend_products_BPR(user_id, K=10):
    if user_id not in id2user:
        return []
    
    #u = user2id[user_id]
    recs = recommend_BPR(user_id, K)

    return [id2item[i] for i in recs]

In [97]:
user_id = 0 #32 #20
recs_BPR = recommend_products_BPR(user_id, K=10)

print("User seen\n", [id2item[i] for i in user_pos_items[user_id]])
print("Recommend\n", recs)

User seen
 ['18177', '28050', '21364', '22308']
Recommend
 ['22294', '6022', '23723', '16799', '25778', '28740', '16485', '11620', '27363', '15847']


In [98]:
show_products(recs_BPR, "RECOMMENDED PRODUCTS (BPR-MF)")


===== RECOMMENDED PRODUCTS (BPR-MF) =====
PID: 22294 | Name: KingSize Big & Tall Pleated Wrinkle Resistant Side Elastic Chinos | Category: Pants | Dept: Men
PID: 6022 | Name: Flared Bell Bottom Knit Spats LegWarmers - Scandinavian Leg Warmer (Grey) | Category: Leggings | Dept: Women
PID: 23723 | Name: Nautica Men's Pea Coat | Category: Outerwear & Coats | Dept: Men
PID: 16799 | Name: Adult Bright Neon Tank Top in 6 Bright Colors | Category: Tops & Tees | Dept: Men
PID: 25778 | Name: Diesel Men's Semaji Boxer | Category: Underwear | Dept: Men
PID: 28740 | Name: Ray-Ban 0RB3497 004/9A Polarized Rimless SunglassesGunmetal59 mm | Category: Accessories | Dept: Men
PID: 16485 | Name: Retro Creeper - Minecraft T-shirt | Category: Tops & Tees | Dept: Men
PID: 11620 | Name: Half Slip Sand24-M | Category: Intimates | Dept: Women
PID: 27363 | Name: Briefly Stated Men's Ready For Love | Category: Sleep & Lounge | Dept: Men
PID: 15847 | Name: Woman Within Plus Size Top in soft knit the Perfect cot

In [99]:
def hit_rate_BPR(K=10):
    hits = 0
    total = 0
    
    for _, row in test_df.iterrows():
        user = row['uid']
        item = row['product_id']
        
        if user not in id2user or item not in item2id:
            continue
        
        i = item2id[item]
        
        recs = recommend_BPR(u, K)
        
        if i in recs:
            hits += 1
        
        total += 1
    return hits / total

def ndcg_at_k_BPR(K=10):
    total = 0
    score = 0
    
    for _, row in test_df.iterrows():
        user = row['uid']
        item = row['product_id']
        
        if user not in id2user or item not in item2id:
            continue
        
        i = item2id[item]
        
        recs = recommend_BPR(u, K)
        
        if i in recs:
            rank = list(recs).index(i)
            score += 1 / np.log2(rank + 2)
        
        total += 1
    return score / total

In [101]:
for K in [5, 10, 20]:
    print(f"HitRate@{K}: {hit_rate_BPR(K):.6f}")
for K in [5, 10, 20]:
    print(f"NDCG@{K}: {ndcg_at_k_BPR(K):.6f}")

HitRate@5: 0.000485
HitRate@10: 0.000646
HitRate@20: 0.001778
NDCG@5: 0.000312
NDCG@10: 0.000370
NDCG@20: 0.000648


**Content-based, Collaborative Consine, Collaborative Matrix Factorization**

In [40]:
#Ma trận tương tác nhị phân
X_binary = csr_matrix((np.ones(len(train_df)), (train_df['uid'], train_df['iid'])),shape=(num_users, num_items))

X_binary.data = np.ones_like(X_binary.data)
print(num_users)
print(num_items)

8063
21727


In [41]:
print("Max:", X_binary.max())
print("Min:", X_binary.min())

Max: 1.0
Min: 0.0


In [42]:
print(X_binary[:5, :10])
print(X_binary[:5, :10].toarray())

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10 stored elements and shape (5, 10)>
  Coords	Values
  (0, 0)	1.0
  (0, 1)	1.0
  (0, 2)	1.0
  (0, 3)	1.0
  (1, 4)	1.0
  (1, 5)	1.0
  (1, 6)	1.0
  (1, 7)	1.0
  (1, 8)	1.0
  (2, 9)	1.0
[[1. 1. 1. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 1. 1. 1. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]


In [43]:
#Ma trận phân rã bằng Truncated SVD
svd = TruncatedSVD(n_components=50)
item_embeddings = svd.fit_transform(X_binary.T)  # (items x dim)
item_embeddings = normalize(item_embeddings)
item_embeddings[2]

array([ 1.08089792e-01, -1.16399580e-01,  8.05153293e-02, -4.47605119e-02,
        1.15045369e-02,  4.62611048e-02, -2.96270283e-01,  7.75615688e-02,
       -2.42056874e-01, -7.73155834e-02, -6.07102077e-02,  2.61302507e-01,
        6.01070393e-02, -1.14734443e-01, -1.32326362e-04,  1.40723164e-01,
       -7.64038567e-02,  4.30295206e-02, -1.90222578e-01,  1.57739773e-02,
       -2.77431117e-01, -1.78370689e-01,  1.55684922e-01,  1.39822131e-01,
        1.43367742e-01,  1.00740754e-01, -1.38826160e-01, -7.90395908e-02,
        3.19074416e-02, -1.16844097e-01,  1.57536113e-01, -1.46481072e-01,
        1.25089646e-01, -1.25408145e-01,  8.39061224e-02,  9.84689570e-02,
        3.48052935e-01,  7.30320312e-03,  4.71946350e-02,  6.56399300e-02,
       -1.43194731e-01, -1.39559579e-01, -4.21234457e-02,  1.82865816e-01,
       -1.23417614e-02,  6.70042374e-02, -1.46145770e-01,  2.14712030e-01,
       -2.10353532e-01,  1.25910266e-01])

In [30]:
#Xây dựng đặc trưng thuộc tính và văn bản của item
#Copy df_products và đảm bảo mỗi product_id unique
item_df = meta.copy()

#Chọn các cột meta cần dùng cho CBF
categorical_cols = ['category', 'brand', 'department']
text_col = 'name'

#One-hot encode các cột categorical
cat_features = pd.get_dummies(item_df[categorical_cols])

# tăng trọng số department (x3)
dept_cols = [col for col in cat_features.columns if col.startswith('department_')]
cat_features[dept_cols] = cat_features[dept_cols] * 3.0

# tăng trọng số category (x2)
cat_cols = [c for c in cat_features.columns if c.startswith('category_')]
cat_features[cat_cols] *= 2

# trọng số brand (x1)
brand_cols = [c for c in cat_features.columns if c.startswith('brand_')]
cat_features[brand_cols] *= 1

# chuyển thành sparse để kết hợp với TF-IDF
cat_features_sparse = sp.csr_matrix(cat_features.astype(float).values)
cat_features_sparse = normalize(cat_features_sparse)

In [31]:
cat_features.head()

,category_Accessories,category_Active,category_Blazers & Jackets,category_Clothing Sets,category_Dresses,category_Fashion Hoodies & Sweatshirts,category_Intimates,category_Jeans,category_Jumpsuits & Rompers,category_Leggings,...,brand_prAna,brand_robesale,brand_stonepowerss,brand_tabbisocks,brand_tasc Performance,brand_turkishtowels,brand_vip boutique,brand_wear ease,department_Men,department_Women
0,0,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,3.0,0.0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,3.0,0.0
2,0,0,0,0,0,0,0,2,0,0,...,0,0,0,0,0,0,0,0,3.0,0.0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,3.0,0.0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,3.0,0.0


In [32]:
#Vector hóa cột name bằng TF-IDF ---
tfidf = TfidfVectorizer(max_features=500)  # giới hạn số chiều vector
name_features = tfidf.fit_transform(item_df[text_col].fillna(''))
item_features = sp.hstack([cat_features_sparse, name_features], format='csr')

# item_features: sparse matrix (num_items x num_features)
num_items_feature = item_features.shape[0]

In [34]:
idx = 0  # sản phẩm đầu tiên
row = item_features[idx]

non_zero_idx = row.nonzero()[1]
values = row.data

for i, v in zip(non_zero_idx, values):
    print(feature_names[i], v)

category_Active 0.5345224838248488
brand_SmartWool 0.2672612419124244
department_Men 0.8017837257372732
smartwool 0.7106960621134658
men 0.24563095170663837
bottom 0.6592241977203992


In [141]:
print(item_features[:5, :10].toarray())

[[0.         0.53452248 0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.53452248 0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]]


In [44]:
def predict_content_based(X_binary, item_features, normalize=True):
    from sklearn.preprocessing import normalize as sk_normalize

    # normalize item features (cosine)
    if normalize:
        item_features = sk_normalize(item_features)

    # build user profile
    user_profiles = X_binary @ item_features  # (num_users x num_features)

    if normalize:
        user_profiles = sk_normalize(user_profiles)

    # compute scores
    scores = user_profiles @ item_features.T  # (num_users x num_items)

    return scores.toarray() if hasattr(scores, "toarray") else scores

def recommend_content_based(X_binary, item_features, top_k=10):
    scores = predict_content_based(X_binary, item_features)

    # remove seen items
    user_ids, item_ids = X_binary.nonzero()
    scores[user_ids, item_ids] = -np.inf

    # top-k
    top_items = np.argsort(-scores, axis=1)[:, :top_k]

    return top_items, scores

In [45]:
#Item-based (CF-based)
def build_cf_similarity(X_binary, top_k=50):

    item_vectors = X_binary.T

    nbrs = NearestNeighbors(n_neighbors=top_k + 1, metric='cosine', algorithm='brute')
    nbrs.fit(item_vectors)

    distances, indices = nbrs.kneighbors(item_vectors)

    # bỏ chính nó
    indices = indices[:, 1:]
    distances = distances[:, 1:]

    sim = 1 - distances

    return indices, sim

def predict_cf_knn(X_binary, indices, sim):
    num_users, num_items = X_binary.shape
    scores = np.zeros((num_users, num_items))

    for item_id in range(num_items):
        neighbors = indices[item_id]    # (top_k,)
        sims = sim[item_id]             # (top_k,)

        # lấy các item user từng tương tác
        neighbor_matrix = X_binary[:, neighbors]  # (num_users, top_k)

        # weighted sum
        scores[:, item_id] = neighbor_matrix.dot(sims)

    return scores

def recommend_cf_knn(X_binary, indices, sim, top_k=10):
    scores = predict_cf_knn(X_binary, indices, sim)
    raw_score = scores.copy()

    # loại item đã tương tác
    user_ids, item_ids = X_binary.nonzero()
    scores[user_ids, item_ids] = -np.inf

    # lấy top-k item cho mỗi user
    top_items = np.argsort(-scores, axis=1)[:, :top_k]

    return top_items, scores, raw_score

In [55]:
def predict_cf_mf(train_df, item_embeddings, num_users, num_items, N=5):
    """
    Item-based recommendation dùng item embeddings (MF/SVD)
    Trả về full score matrix (num_users x num_items)
    """

    # normalize embedding (cosine)
    item_embeddings = item_embeddings / (
        np.linalg.norm(item_embeddings, axis=1, keepdims=True) + 1e-8
    )

    scores = np.zeros((num_users, num_items))

    # lấy last N items mỗi user
    user_items = (
        train_df.sort_values('created_at')
        .groupby('uid')['iid']
        .apply(lambda x: x.tail(N).tolist())
    )

    for uid, items in user_items.items():
        if len(items) == 0:
            continue

        item_vecs = item_embeddings[items]              # (N x dim)
        sims = item_vecs @ item_embeddings.T            # (N x num_items)

        # aggregate (mean)
        scores[uid] = np.mean(sims, axis=0)

    return scores

def recommend_cf_mf(train_df, item_embeddings, num_users, num_items, top_k=10, N=5):
    scores = predict_cf_mf(train_df, item_embeddings, num_users, num_items, N)

    # remove seen items
    user_ids = train_df['uid'].values
    item_ids = train_df['iid'].values
    scores[user_ids, item_ids] = -np.inf

    # top-k
    top_items = np.argsort(-scores, axis=1)[:, :top_k]

    return top_items, scores

In [47]:
user_id = 0

In [48]:
show_user_seen_products(uid=user_id,df_copy=df_copy,meta_products=meta_products,id2user=id2user,top_n=10)


===== USER 0 (real id: 3.0) — SEEN PRODUCTS =====
PID: 18177 | Name: Smartwool Men's Midweight Bottom | Category: Active | Dept: Men
PID: 28050 | Name: Volcom Men's Bruce Annihilator Board-Shorts | Category: Swim | Dept: Men
PID: 21364 | Name: Marc Ecko Cut & Sew Men's Baked Alaska Bootcut Fit Jean | Category: Jeans | Dept: Men
PID: 22308 | Name: Dockers Men's Limited Offer D2 Stretch Khaki Pant | Category: Pants | Dept: Men
PID: 26696 | Name: Tommy Bahama Terry Loop Robe | Category: Sleep & Lounge | Dept: Men


In [51]:
top_item_ct, score_ct = recommend_content_based(X_binary, item_features, top_k=10)

In [52]:
recommended_encoded_items = top_item_ct[user_id]
recommended_product_ids = [id2item[iid] for iid in recommended_encoded_items]
show_products(recommended_product_ids, "RECOMMENDED PRODUCTS (Content-Based)")


===== RECOMMENDED PRODUCTS (Content-Based) =====
PID: 28186 | Name: Rip Curl Mirage Flex Accelerate | Category: Swim | Dept: Men
PID: 22552 | Name: Dockers Men's Limited Offer Explorer Khaki Pant | Category: Pants | Dept: Men
PID: 22385 | Name: Dockers Men's Limited Offer D1 Anchor Khaki Pant | Category: Pants | Dept: Men
PID: 21792 | Name: Dockers Men's Limited Offer D2 Straight Fit Khaki Pant | Category: Pants | Dept: Men
PID: 21967 | Name: Dockers Men's Coated Alpha Khaki Slim Fit Pant | Category: Pants | Dept: Men
PID: 22100 | Name: Dockers Men's Limited Offer D1 Slim Fit Sateen Khaki Pant | Category: Pants | Dept: Men
PID: 22291 | Name: Dockers Men's Limited Offer D1 Slim Fit Hollywood Khaki Pant | Category: Pants | Dept: Men
PID: 27766 | Name: Volcom Men's Bosca Board Short | Category: Swim | Dept: Men
PID: 27626 | Name: Volcom Men's Dredge Board Short | Category: Swim | Dept: Men
PID: 27850 | Name: Volcom Men's Gunshot Board Short | Category: Swim | Dept: Men


In [53]:
indices_cf, sim_cf = build_cf_similarity(X_binary, top_k=50)
top_items_cf, scores_cf, raw_score_4 = recommend_cf_knn(X_binary,indices_cf,sim_cf,top_k=10)

In [54]:
recommended_encoded_items = top_items_cf[user_id]
recommended_product_ids = [id2item[iid] for iid in recommended_encoded_items]
show_products(recommended_product_ids, "RECOMMENDED PRODUCTS (Collabortive Consine Similarity)")


===== RECOMMENDED PRODUCTS (Collabortive Consine Similarity) =====
PID: 22574 | Name: American Apparel Stretch Chambray Stripe Travel Pant | Category: Pants | Dept: Men
PID: 23523 | Name: DICKIES Mens Relaxed Fit Shorts | Category: Shorts | Dept: Men
PID: 19559 | Name: Diesel Men's K-Bolka Sweater | Category: Sweaters | Dept: Men
PID: 28879 | Name: Oakley Men's Straight Jacket Iridium Polarized Sunglasses | Category: Accessories | Dept: Men
PID: 23756 | Name: London Fog Men's Vista Systems Parka Coat | Category: Outerwear & Coats | Dept: Men
PID: 23344 | Name: Union Skyway Cargo Shorts (450H) - Red | Category: Shorts | Dept: Men
PID: 28584 | Name: Guide Gear Leather Rabbit Fur Hat | Category: Accessories | Dept: Men
PID: 23416 | Name: Ed Garments Men's Flat Front Utility Cargo Short. 2468 | Category: Shorts | Dept: Men
PID: 21808 | Name: Mountain Khakis Men's Lake Lodge Twill Pant | Category: Pants | Dept: Men
PID: 26127 | Name: Clever Men's Victoria Thong | Category: Underwear | Dept

In [56]:
top_items_cf_2, scores_cf_2 = recommend_cf_mf(train_df, item_embeddings, num_users, num_items, top_k=10, N=5)

In [58]:
recommended_encoded_items = top_items_cf_2[user_id]
recommended_product_ids = [id2item[iid] for iid in recommended_encoded_items]
show_products(recommended_product_ids, "RECOMMENDED PRODUCTS (Collaborative Matrix Factorization)")


===== RECOMMENDED PRODUCTS (Collaborative Matrix Factorization) =====
PID: 19559 | Name: Diesel Men's K-Bolka Sweater | Category: Sweaters | Dept: Men
PID: 23523 | Name: DICKIES Mens Relaxed Fit Shorts | Category: Shorts | Dept: Men
PID: 16803 | Name: Enro Non-Iron Pinpoint Oxford Dress Shirt | Category: Tops & Tees | Dept: Men
PID: 28584 | Name: Guide Gear Leather Rabbit Fur Hat | Category: Accessories | Dept: Men
PID: 17769 | Name: DC Men's Star Pullover Hoody | Category: Fashion Hoodies & Sweatshirts | Dept: Men
PID: 21808 | Name: Mountain Khakis Men's Lake Lodge Twill Pant | Category: Pants | Dept: Men
PID: 28878 | Name: Levi's Brown Tonal Stitched Money Clip Bifold Wallet | Category: Accessories | Dept: Men
PID: 17866 | Name: Caterpillar Mens Tm Banner Hoodie | Category: Fashion Hoodies & Sweatshirts | Dept: Men
PID: 20313 | Name: Men's Superior 150's Single Breasted One Button Brown Vested Dress Suit with Wide Leg Pants | Category: Suits & Sport Coats | Dept: Men
PID: 21553 | Na

In [104]:
from collections import defaultdict
def build_ground_truth(test_df):
    gt = defaultdict(set)

    for _, row in test_df.iterrows():
        gt[row['uid']].add(row['iid'])

    return gt

In [105]:
def hitrate_at_k_single(top_item_idx, ground_truth, k=10):
    precisions = []

    num_users = top_item_idx.shape[0]

    for uid in range(num_users):
        if uid not in ground_truth:
            continue

        recs = top_item_idx[uid][:k]
        gt = ground_truth[uid]

        hits = sum(1 for item in recs if item in gt)
        precisions.append(hits / k)

    return np.mean(precisions) if precisions else 0.0

In [110]:
def dcg_single(recommended, true_items, K):
    dcg_score = 0

    for i, item in enumerate(recommended[:K]):
        if item in true_items:
            dcg_score += 1 / np.log2(i + 2)

    return dcg_score

def ndcg_k_single(top_k_items, test_df, K=10):
    gt = build_ground_truth(test_df)

    ndcg_sum = 0
    users = 0

    for uid in gt:
        ideal_dcg = sum(1 / np.log2(i + 2) for i in range(min(len(gt[uid]), K)))
        actual_dcg = dcg_single(top_k_items[uid], gt[uid], K)

        ndcg_sum += actual_dcg / (ideal_dcg + 1e-8)
        users += 1

    return ndcg_sum / users

In [111]:
def evaluate_advanced_single(top_items_idx, test_df, item_embeddings, K=10):

    print("HitRate@K:",hitrate_at_k_single(top_items_idx, build_ground_truth(test_df), K))
    print("NDCG@K:",ndcg_k_single(top_items_idx, test_df, K))

In [112]:
K = 10 
print("Collaborative Matrix Factorization")
evaluate_advanced_single(top_items_cf_2, test_df, item_embeddings, K=10)

Collaborative Matrix Factorization
HitRate@K: 4.8480930833872014e-05
NDCG@K: 0.00020158768803279272


In [113]:
print("Collaborative Consine")
evaluate_advanced_single(top_items_cf, test_df, item_embeddings, K=10)

Collaborative Consine
HitRate@K: 3.2320620555914674e-05
NDCG@K: 0.00010196020479349711


In [114]:
print("Content-based")
evaluate_advanced_single(top_item_ct, test_df, item_embeddings, K=10) 

Content-based
HitRate@K: 8.080155138978668e-05
NDCG@K: 0.0004234352056096603


In [115]:
def ratio_at_k_single(top_items, ground_truth, item_meta, id2item, k, col='category'):

    total_ratio = 0
    valid_users = 0
    total_match = 0
    total_items_checked = 0

    meta_dict = item_meta[col].to_dict()

    for u in range(top_items.shape[0]):
        gt_items = ground_truth.get(u)
        if not gt_items:
            continue

        gt_encoded = gt_items[0]
        gt_real = id2item.get(gt_encoded)

        if gt_real not in meta_dict:
            continue

        gt_value = meta_dict[gt_real]

        rec_real = [id2item[i] for i in top_items[u, :k]]

        match_count = sum(
            1 for item in rec_real
            if meta_dict.get(item) == gt_value
        )

        total_ratio += match_count / k
        total_items_checked += k
        total_match += match_count
        valid_users += 1

    avg_ratio = total_ratio / valid_users if valid_users > 0 else 0

    print("User in test:", valid_users)
    print("Total check in recommend:", total_items_checked)
    print("Total match:", total_match)

    return avg_ratio

In [117]:
item_meta_hyb = meta_products.set_index('id')[['category', 'department']]

In [118]:
k = 10 #check 10 recommendation đầu tiên

#đếm bao nhiêu item trong top-10 có cùng category/department với ground truth item
cat_ratio = ratio_at_k_single(top_item_ct, ground_truth, item_meta_hyb, id2item, k=10, col='category')
print(f"CategoryRatio: {cat_ratio:.4f}\n")

dept_ratio = ratio_at_k_single(top_item_ct, ground_truth, item_meta_hyb, id2item, k=10, col='department')
print(f"DepartmentRatio: {dept_ratio:.4f}")

User in test: 6188
Total check in recommend: 61880
Total match: 5874
CategoryRatio: 0.0949

User in test: 6188
Total check in recommend: 61880
Total match: 61880
DepartmentRatio: 1.0000


In [320]:
k = 10 #check 10 recommendation đầu tiên

#đếm bao nhiêu item trong top-10 có cùng category/department với ground truth item
cat_ratio = ratio_at_k_single(top_items_cf, ground_truth, item_meta_hyb, id2item, k=10, col='category')
print(f"CategoryRatio: {cat_ratio:.4f}\n")

dept_ratio = ratio_at_k_single(top_items_cf, ground_truth, item_meta_hyb, id2item, k=10, col='department')
print(f"DepartmentRatio: {dept_ratio:.4f}")

User in test: 871
Total check in recommend: 8710
Total match: 545
CategoryRatio: 0.0626

User in test: 871
Total check in recommend: 8710
Total match: 8388
DepartmentRatio: 0.9630


In [319]:
k = 10 #check 10 recommendation đầu tiên

#đếm bao nhiêu item trong top-10 có cùng category/department với ground truth item
cat_ratio = ratio_at_k_single(top_items_cf_2, ground_truth, item_meta_hyb, id2item, k=10, col='category')
print(f"CategoryRatio: {cat_ratio:.4f}\n")

dept_ratio = ratio_at_k_single(top_items_cf_2, ground_truth, item_meta_hyb, id2item, k=10, col='department')
print(f"DepartmentRatio: {dept_ratio:.4f}")

User in test: 871
Total check in recommend: 8710
Total match: 546
CategoryRatio: 0.0627

User in test: 871
Total check in recommend: 8710
Total match: 8504
DepartmentRatio: 0.9763


**Hybrid Recommendation**

In [72]:
def hybrid_scores(als_model,X_binary, train_df, item_features, item_embeddings, num_users,num_items):
    
    # --- ALS ---
    als_scores = als_model.X @ als_model.Y.T

    # --- Content-based ---
    content_scores = predict_content_based(X_binary, item_features)

    # --- CF KNN ---
    cf_scores = predict_cf_mf(train_df, item_embeddings, num_users, num_items, N=5)

    # --- normalize về cùng scale ---
    def normalize_hybrid(x):
        return (x - x.min()) / (x.max() - x.min() + 1e-8)

    als_scores = normalize_hybrid(als_scores)
    content_scores = normalize_hybrid(content_scores)
    cf_scores = normalize_hybrid(cf_scores)

    # --- combine ---
    final_scores = 0.4 * als_scores + 0.4 * cf_scores + 0.2 * content_scores

    return final_scores

In [73]:
def recommend_hybrid(als_model,X_binary,train_df,item_features, item_embeddings, num_users,num_items,top_k=10):
    
    scores = hybrid_scores(als_model,X_binary,train_df,item_features, item_embeddings, num_users,num_items)

    # remove seen items
    user_ids, item_ids = X_binary.nonzero()
    scores[user_ids, item_ids] = -np.inf

    top_items = np.argsort(-scores, axis=1)[:, :top_k]

    return top_items, scores

In [120]:
def hitrate_at_k_hybrid(top_items, ground_truth):
    hits = 0
    num_users = top_items.shape[0]

    for u in range(num_users):
        gt = ground_truth.get(u, [])
        if any(item in top_items[u] for item in gt):
            hits += 1

    return hits / num_users

def ndcg_at_k_hybrid(top_items, ground_truth, k):
    import numpy as np

    def dcg(rel):
        return np.sum([r / np.log2(i + 2) for i, r in enumerate(rel)])

    total_ndcg = 0
    num_users = top_items.shape[0]

    for u in range(num_users):
        gt = set(ground_truth.get(u, []))

        rel = [1 if item in gt else 0 for item in top_items[u][:k]]

        dcg_val = dcg(rel)
        idcg_val = dcg(sorted(rel, reverse=True))

        if idcg_val > 0:
            total_ndcg += dcg_val / idcg_val

    return total_ndcg / num_users

In [75]:
ground_truth = test_df.groupby('uid')['iid'].apply(list).to_dict()

In [121]:
top_k = 5

top_items_hybrid, scores_hybrid = recommend_hybrid(model_ALS,X_binary,train_df,item_features, item_embeddings, num_users,num_items,top_k=top_k)

hr = hitrate_at_k_hybrid(top_items_hybrid, ground_truth)
ndcg = ndcg_at_k_hybrid(top_items_hybrid, ground_truth, top_k)

print(f"HR@{top_k}: {hr:.6f}")
print(f"NDCG@{top_k}: {ndcg:.6f}")

HR@5: 0.000124
NDCG@5: 0.000078


In [122]:
top_k = 10

top_items_hybrid, scores_hybrid = recommend_hybrid(model_ALS,X_binary,train_df,item_features, item_embeddings, num_users,num_items,top_k=top_k)

hr = hitrate_at_k_hybrid(top_items_hybrid, ground_truth)
ndcg = ndcg_at_k_hybrid(top_items_hybrid, ground_truth, top_k)

print(f"HR@{top_k}: {hr:.6f}")
print(f"NDCG@{top_k}: {ndcg:.6f}")

HR@10: 0.000496
NDCG@10: 0.000199


In [123]:
top_k = 15

top_items_hybrid, scores_hybrid = recommend_hybrid(model_ALS,X_binary,train_df,item_features, item_embeddings, num_users,num_items,top_k=top_k)

hr = hitrate_at_k_hybrid(top_items_hybrid, ground_truth)
ndcg = ndcg_at_k_hybrid(top_items_hybrid, ground_truth, top_k)

print(f"HR@{top_k}: {hr:.6f}")
print(f"NDCG@{top_k}: {ndcg:.6f}")

HR@15: 0.001116
NDCG@15: 0.000368


In [78]:
def recommend_for_user(user_id, scores, top_k=10):
    return np.argsort(-scores[user_id])[:top_k]

In [79]:
user_id = 0
recs_hybrid = recommend_for_user(user_id, scores_hybrid, top_k=10)

show_user_seen_products(uid=user_id,df_copy=df_copy,meta_products=meta_products,id2user=id2user,top_n=10)
recommended_product_ids = [id2item[iid] for iid in recs_hybrid]
show_products(recommended_product_ids, "RECOMMENDED PRODUCTS (Hybrid)")


===== USER 0 (real id: 3.0) — SEEN PRODUCTS =====
PID: 18177 | Name: Smartwool Men's Midweight Bottom | Category: Active | Dept: Men
PID: 28050 | Name: Volcom Men's Bruce Annihilator Board-Shorts | Category: Swim | Dept: Men
PID: 21364 | Name: Marc Ecko Cut & Sew Men's Baked Alaska Bootcut Fit Jean | Category: Jeans | Dept: Men
PID: 22308 | Name: Dockers Men's Limited Offer D2 Stretch Khaki Pant | Category: Pants | Dept: Men
PID: 26696 | Name: Tommy Bahama Terry Loop Robe | Category: Sleep & Lounge | Dept: Men

===== RECOMMENDED PRODUCTS (Hybrid) =====
PID: 21553 | Name: Mavi Men's Daniel Lowrise Jean | Category: Jeans | Dept: Men
PID: 21808 | Name: Mountain Khakis Men's Lake Lodge Twill Pant | Category: Pants | Dept: Men
PID: 17866 | Name: Caterpillar Mens Tm Banner Hoodie | Category: Fashion Hoodies & Sweatshirts | Dept: Men
PID: 23523 | Name: DICKIES Mens Relaxed Fit Shorts | Category: Shorts | Dept: Men
PID: 28878 | Name: Levi's Brown Tonal Stitched Money Clip Bifold Wallet | Cate

In [125]:
def ratio_at_k_hybrid(top_items, ground_truth, item_meta, id2item, k, col='category'):

    total_ratio = 0
    valid_users = 0
    total_match = 0
    total_items_checked = 0

    meta_dict = item_meta[col].to_dict()

    for u in range(top_items.shape[0]):
        gt_items = ground_truth.get(u)
        if not gt_items:
            continue

        gt_encoded = gt_items[0]
        gt_real = id2item.get(gt_encoded)

        if gt_real not in meta_dict:
            continue

        gt_value = meta_dict[gt_real]

        rec_real = [id2item[i] for i in top_items[u, :k]]

        match_count = sum(
            1 for item in rec_real
            if meta_dict.get(item) == gt_value
        )

        total_ratio += match_count / k
        total_items_checked += k
        total_match += match_count
        valid_users += 1

    avg_ratio = total_ratio / valid_users if valid_users > 0 else 0

    print("User in test:", valid_users)
    print("Total check in recommend:", total_items_checked)
    print("Total match:", total_match)

    return avg_ratio

In [126]:
k = 5 #check 5 recommendation đầu tiên

#đếm bao nhiêu item trong top-10 có cùng category/department với ground truth item
cat_ratio = ratio_at_k_hybrid(top_items_hybrid, ground_truth, item_meta_hyb, id2item, k=5, col='category')
print(f"CategoryRatio: {cat_ratio:.4f}\n")

dept_ratio = ratio_at_k_hybrid(top_items_hybrid, ground_truth, item_meta_hyb, id2item, k=5, col='department')
print(f"DepartmentRatio: {dept_ratio:.4f}")

User in test: 6188
Total check in recommend: 30940
Total match: 2179
CategoryRatio: 0.0704

User in test: 6188
Total check in recommend: 30940
Total match: 30940
DepartmentRatio: 1.0000


In [128]:
k = 10 #check 10 recommendation đầu tiên

#đếm bao nhiêu item trong top-10 có cùng category/department với ground truth item
cat_ratio = ratio_at_k_hybrid(top_items_hybrid, ground_truth, item_meta_hyb, id2item, k=10, col='category')
print(f"CategoryRatio: {cat_ratio:.4f}\n")

dept_ratio = ratio_at_k_hybrid(top_items_hybrid, ground_truth, item_meta_hyb, id2item, k=10, col='department')
print(f"DepartmentRatio: {dept_ratio:.4f}")

User in test: 6188
Total check in recommend: 61880
Total match: 4230
CategoryRatio: 0.0684

User in test: 6188
Total check in recommend: 61880
Total match: 61880
DepartmentRatio: 1.0000


In [127]:
k = 15 #check 10 recommendation đầu tiên

#đếm bao nhiêu item trong top-10 có cùng category/department với ground truth item
cat_ratio = ratio_at_k_hybrid(top_items_hybrid, ground_truth, item_meta_hyb, id2item, k=15, col='category')
print(f"CategoryRatio: {cat_ratio:.4f}\n")

dept_ratio = ratio_at_k_hybrid(top_items_hybrid, ground_truth, item_meta_hyb, id2item, k=15, col='department')
print(f"DepartmentRatio: {dept_ratio:.4f}")

User in test: 6188
Total check in recommend: 92820
Total match: 6345
CategoryRatio: 0.0684

User in test: 6188
Total check in recommend: 92820
Total match: 92820
DepartmentRatio: 1.0000


In [130]:
print(train_df.shape)
print(test_df.shape)


(49649, 16)
(6188, 16)
